In [1]:
%pip install -U langchain langchain-core langchain-community langchain-openai langchain-openrouter langchain_huggingface langchain-qdrant langchain-text-splitters qdrant-client python-dotenv pypdf pymupdf

  Using cached langchain_huggingface-1.2.3-py3-none-any.whl.metadata (4.3 kB)
  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
Using cached langchain_huggingface-1.2.3-py3-none-any.whl (34 kB)
Using cached huggingface_hub-1.33.0-py3-none-any.whl (846 kB)

  Attempting uninstall: huggingface-hub

    Found existing installation: huggingface_hub 2.2.0

   ---------------------------------------- 0/2 [huggingface-hub]
    Uninstalling huggingface_hub-2.2.0:
   ---------------------------------------- 0/2 [huggingface-hub]
      Successfully uninstalled huggingface_hub-2.2.0
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   

# Loading Documents

In [2]:
from langchain_core.documents import Document

doc = Document(
    page_content=(
        "Full-time employees receive 24 days of paid annual leave per calendar year. "
        "Up to 5 unused days may be carried over into the next year."
    ),
    metadata={"filename": "employee_handbook.pdf", "page": 1, "area": "policies"},
)

print(doc.page_content)
print(doc.metadata)

Full-time employees receive 24 days of paid annual leave per calendar year. Up to 5 unused days may be carried over into the next year.
{'filename': 'employee_handbook.pdf', 'page': 1, 'area': 'policies'}


## Loading text and Markdown

In [3]:
from langchain_community.document_loaders import TextLoader, UnstructuredMarkdownLoader

notes = TextLoader("knowledge/wiki/onboarding.md", encoding="utf-8").load()

print(len(notes))
print(notes[0].metadata)
print(notes[0].page_content)




C:\Users\Beher\AppData\Local\Temp\ipykernel_22212\737010498.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, UnstructuredMarkdownLoader


1
{'source': 'knowledge/wiki/onboarding.md'}
# Engineering Onboarding

Welcome to Zomato Engineering. This page covers your first weeks. Ask in #new-joiners if anything is unclear.

## Day one

- Collect your laptop and access card from the IT desk.
- Meet your onboarding buddy. They will walk you through the team's codebase and rituals.
- Complete the mandatory security awareness course in the Learning Portal within your first 3 days.
- Join the team Slack channels listed in your welcome email.

## Accounts and access

You will receive these accounts on day one: company email, Slack, GitHub, Jira, Confluence and the internal Zomato Admin dashboard. Production access is **not** granted automatically. Request it through the Access Portal after you finish the security course; your manager must approve it, and access is reviewed every quarter.

### VPN setup

Install the company VPN client from the IT portal, then sign in with your work email and the authenticator app. Connect to the VPN 

In [4]:
%pip install unstructured
%pip install markdown

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [5]:
notes_md = UnstructuredMarkdownLoader("knowledge/wiki/onboarding.md", mode="elements").load()

print(len(notes_md))
print(notes_md[0])

44
page_content='Engineering Onboarding' metadata={'source': 'knowledge/wiki/onboarding.md', 'category_depth': 0, 'languages': ['eng'], 'file_directory': 'knowledge/wiki', 'filename': 'onboarding.md', 'filetype': 'text/markdown', 'last_modified': '2026-10-08T20:29:36', 'category': 'Title', 'element_id': '5538e4f54984a9265fafa7d6d6b4943b'}


## Loading a PDF

In [6]:
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader("knowledge/policies/employee_handbook.pdf").load()

print(f"total pages: {len(pages)}")

for i, p in enumerate(pages):
    print(f"page: {i+1} \n{p}")

total pages: 8
page: 1 
page_content='Zomato Employee Handbook
Version 4.2 · Effective 1 April 2026 · Owner: People Operations
1. Welcome to Zomato
Zomato connects millions of customers with restaurants and delivery partners every day. We are a technology company at
heart, and our work touches food, logistics, payments and support. This handbook explains how we work, what you can
expect from us and what we expect from you. If anything here conflicts with your employment contract, the contract
applies.
2. Our values
Customer obsession. Start with what the customer, restaurant partner or delivery partner needs, then work
backwards.
Ownership. If you see a problem, you own it until it is fixed or handed over clearly.
Speed with care. We move fast, and we do not cut corners on food safety, payments or privacy.
Simple and honest. Say what you mean, share bad news early and prefer plain language over jargon.
Respect for everyone. Every person who works with Zomato, from a rider to a director

In [7]:
%pip install pdfplumber

Note: you may need to restart the kernel to use updated packages.


In [8]:
import time
from langchain_community.document_loaders import PDFPlumberLoader, PyMuPDFLoader, PyPDFLoader

path = "knowledge/policies/expense_policy.pdf"      # a PDF that contains a table
candidates = {"pypdf": PyPDFLoader, "pymupdf": PyMuPDFLoader, "pdfplumber": PDFPlumberLoader}

for name, loader_cls in candidates.items():
    started = time.perf_counter()
    pages = loader_cls(path).load()
    seconds = time.perf_counter() - started
    chars = sum(len(p.page_content) for p in pages)
    print(f"{name:>10}: {len(pages)} pages, {chars:,} characters, {seconds:.2f}s")
    print("            sample:", pages[2].page_content[:90].replace("\n", " | "))

     pypdf: 5 pages, 6,239 characters, 0.12s
            sample: 6. Domestic travel | Item Rule | Flights Economy class. Book at least 7 days ahead where possi
   pymupdf: 5 pages, 6,239 characters, 0.67s
            sample: 6. Domestic travel | Item | Rule | Flights | Economy class. Book at least 7 days ahead where possi
pdfplumber: 5 pages, 6,244 characters, 0.71s
            sample: 6. Domestic travel | Item Rule | Flights Economy class. Book at least 7 days ahead where possi


### Scanned PDFs need OCR

In [9]:
%pip install "unstructured[pdf]"
# OCR also needs system tools: tesseract and poppler (for example: brew install tesseract poppler)

Note: you may need to restart the kernel to use updated packages.


In [10]:
from langchain_community.document_loaders import UnstructuredPDFLoader
import os
import shutil

os.environ["PATH"] = (
    r"C:\Program Files\Tesseract-OCR"
    + os.pathsep
    + os.environ.get("PATH", "")
)
print(shutil.which("tesseract"))  # should show the path to tesseract.exe

scan = UnstructuredPDFLoader(
    "knowledge/scans/vendor_contract.pdf",
    # mode="elements",            # titles, paragraphs and tables as separate Documents
    strategy="ocr_only",        # force OCR instead of reading a text layer
).load()

print(len(scan), "elements")
print(scan[0].metadata.get("category"), "|", scan[0].page_content)

C:\Program Files\Tesseract-OCR\tesseract.EXE


d:\coding shuttle\ai engineer\practice-lab\ai-eng\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No languages specified, defaulting to English.


1 elements
None | MASTER SERVICES AGREEMENT

between ZOMATO LIMITED (“Zomato”) and FRESHROUTE LOGISTICS PRIVATE LIMITED (“Vendor”)

This Master Services Agreement (the “Agreement”) is made on 12 January 2026 at Gurugram between Zomato Limited, having its registered office in Gurugram, Haryana, and FreshRoute Logistics Private Limited, having its registered office in Bengaluru, Karnataka. The Agreement governs the supply of last- mile cold-chain delivery services by the Vendor to Zomato,

1. Definitions

“Services” means the temperature-controlled delivery services described in Clause 3 and Schedule A. “Order” means a single delivery request placed by Zomato. “Promised Window” means the delivery time shown to the customer at checkout. “Monthly Fee” means all fees invoiced for Services delivered in a calendar month, “Confidential Information” has the meaning given in Clause 9.

2. Term

This Agreement starts on | February 2026 and continues for 24 months, unless terminated earlier under 

## CSV, JSON, Word and web loaders

### CSV: one row, one Document

In [11]:
from langchain_community.document_loaders import CSVLoader

faq = CSVLoader(
    "knowledge/faq/support_faq.csv",
    encoding="utf-8",
    metadata_columns=["product", "category"],
).load()

print(len(faq), "rows")
print(faq[0].page_content)
print(faq[0].metadata)

63 rows
question: How do I get a refund for a cancelled order?
answer: Refunds for cancelled orders go back to the original payment method within 5 to 7 working days. Zomato credits are refunded instantly.
{'source': 'knowledge/faq/support_faq.csv', 'row': 0, 'product': 'Zomato App', 'category': 'Refunds'}


### JSON: choose what becomes text

In [12]:
%pip install jq

Note: you may need to restart the kernel to use updated packages.


In [13]:

from langchain_community.document_loaders import JSONLoader

# ".plans[]" -> one Document per plan (3 plans = 3 Documents)
plans = JSONLoader(
    file_path="knowledge/catalog/plans.json",
    jq_schema=".plans[]",
    text_content=False,          # keep each plan as JSON text
).load()

print(len(plans), "documents")   # 3
print(plans[0].page_content)     # the Starter plan, as JSON

# ".plans[] | {name, sso}" -> each Document keeps only those two fields
compact = JSONLoader(
    file_path="knowledge/catalog/plans.json",
    jq_schema=".plans[] | {name, sso}",
    text_content=False,
).load()

# ".billing" -> a single Document holding just the billing block
billing = JSONLoader(
    file_path="knowledge/catalog/plans.json",
    jq_schema=".billing",
    text_content=False,
).load()

# metadata_func: lift a field into metadata so you can filter on it later
def plan_meta(record: dict, metadata: dict) -> dict:
    metadata["plan"] = record.get("name")
    return metadata

per_plan = JSONLoader(
    file_path="knowledge/catalog/plans.json",
    jq_schema=".plans[]",
    text_content=False,
    metadata_func=plan_meta,
).load()
print(per_plan[1].metadata["plan"])   # Business

3 documents
{"name": "Starter", "price_per_employee_per_month": 99, "min_employees": 10, "sso": false, "scim": false, "support": "Email support, response within 2 business days", "features": ["Company meal wallet", "Daily spending limits", "Monthly consolidated invoice", "Email support"]}
Business


## Word, HTML and live web pages

In [14]:
%pip install docx2txt

Note: you may need to restart the kernel to use updated packages.


In [15]:
from langchain_community.document_loaders import BSHTMLLoader, Docx2txtLoader, WebBaseLoader

memo = Docx2txtLoader("knowledge/policies/travel_memo.docx").load()
page = BSHTMLLoader("knowledge/wiki/holiday_calendar.html").load()
site = WebBaseLoader("https://example.com/help/shipping-rates").load()

print(memo)
print(page)
print(site)

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'knowledge/policies/travel_memo.docx'}, page_content='Memo: Changes to business travel booking from 1 May 2026\n\nFrom: Finance and Admin | To: All employees | Date: 15 April 2026\n\nWhy we are making this change\n\nTravel spend has grown 40% over the last year, and many bookings were made directly with airlines and hotels at higher prices. A single booking channel lets us negotiate corporate fares and gives employees support if plans change.\n\nWhat changes\n\nEffective 1 May 2026, all domestic flights and hotels must be booked through the Travel Desk portal instead of directly with airlines or hotels. Train tickets can still be booked on the railway site and claimed as usual.\n\nBookings made outside the portal will not be reimbursed unless the Travel Desk was unavailable and Finance approved the exception in writing.\n\nBooking rules\n\nFlights remain economy class. Book at least 7 days ahead to get the corporate fare. Hotels must stay within the nightl

## Loading a whole folder

In [16]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

loader = DirectoryLoader(
    "knowledge/policies",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True,
)
docs = loader.load()

100%|██████████| 2/2 [00:00<00:00, 40.77it/s]


In [17]:
from ai_eng.settings import KNOWLEDGE_DIR
from ai_eng.loaders import load_corpus

docs = load_corpus(KNOWLEDGE_DIR)

No languages specified, defaulting to English.


Loaded 134 documents from D:\coding shuttle\ai engineer\practice-lab\ai-eng\src\ai_eng\4_langchain_rag\knowledge


# Chuncking Documents

## Splitting into chunks

In [22]:
from ai_eng.settings import KNOWLEDGE_DIR
from ai_eng.loaders import load_corpus

docs = load_corpus(KNOWLEDGE_DIR)

d:\coding shuttle\ai engineer\practice-lab\ai-eng\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No languages specified, defaulting to English.


Loaded 134 documents from D:\coding shuttle\ai engineer\practice-lab\ai-eng\src\ai_eng\4_langchain_rag\knowledge


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,         # maximum characters per chunk
    chunk_overlap=120,      # characters shared between neighbouring chunks
    add_start_index=True,   # remembers where in the page each chunk began
)

chunks = splitter.split_documents(docs)

print(f"Split {len(docs)} documents into {len(chunks)} chunks")
print(chunks[0].page_content[:200])
print(chunks[0].metadata)

Split 134 documents into 158 chunks
question: How do I get a refund for a cancelled order?
answer: Refunds for cancelled orders go back to the original payment method within 5 to 7 working days. Zomato credits are refunded instantly.
{'source': 'D:\\coding shuttle\\ai engineer\\practice-lab\\ai-eng\\src\\ai_eng\\4_langchain_rag\\knowledge\\faq\\support_faq.csv', 'row': 0, 'product': 'Zomato App', 'category': 'Refunds', 'filename': 'support_faq.csv', 'area': 'faq', 'doc_type': 'csv', 'start_index': 0}


In [20]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

text = open("knowledge/wiki/onboarding.md", encoding="utf-8").read()

# 1. cuts only at blank lines, then merges paragraphs up to the size limit
by_char = CharacterTextSplitter(separator="\n\n", chunk_size=500, chunk_overlap=50)

# 2. tries blank lines, then newlines, then spaces, so oversized paragraphs still get split
by_recursive = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# 3. sizes in tokens, matching what the embedding model actually counts
by_tokens = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=120, chunk_overlap=15
)

for name, splitter in [("character", by_char), ("recursive", by_recursive), ("tokens", by_tokens)]:
    pieces = splitter.split_text(text)
    longest = max(len(p) for p in pieces)
    print(f"{name:>10}: {len(pieces):>3} chunks, longest = {longest} characters")

 character:  15 chunks, longest = 479 characters
 recursive:  15 chunks, longest = 479 characters
    tokens:  12 chunks, longest = 596 characters


In [21]:
%pip install tiktoken
import tiktoken            # pip install tiktoken

enc = tiktoken.get_encoding("cl100k_base")      # the encoding used by text-embedding-3-small
ids = enc.encode("Zomato delivers unbelievably fast.")

print(ids)                                  # [57, 316, 4428, 28421, 40037, 89234, 5043, 13]
print([enc.decode([i]) for i in ids])       # ['Z', 'om', 'ato', ' delivers', ' unbelie', 'vably', ' fast', '.']
print(len(ids), "tokens")                   # 8 tokens

Note: you may need to restart the kernel to use updated packages.
[57, 316, 4428, 28421, 40037, 89234, 5043, 13]
['Z', 'om', 'ato', ' delivers', ' unbelie', 'vably', ' fast', '.']
8 tokens


## Header-aware splitting for Markdown and HTML

In [22]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")],
)

markdown = open("knowledge/wiki/onboarding.md", encoding="utf-8").read()
sections = header_splitter.split_text(markdown)

print(len(sections), "sections")
print(sections[2].metadata)
print(sections[2].page_content[:150])

17 sections
{'h1': 'Engineering Onboarding', 'h2': 'Accounts and access'}
You will receive these accounts on day one: company email, Slack, GitHub, Jira, Confluence and the internal Zomato Admin dashboard. Production access 


In [23]:
size_cap = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=120, add_start_index=True)
chunks = size_cap.split_documents(sections)      # sections that already fit stay as they are
for chunk in chunks:
    print(f"{chunk}\n")

page_content='Welcome to Zomato Engineering. This page covers your first weeks. Ask in #new-joiners if anything is unclear.' metadata={'h1': 'Engineering Onboarding', 'start_index': 0}

page_content='- Collect your laptop and access card from the IT desk.
- Meet your onboarding buddy. They will walk you through the team's codebase and rituals.
- Complete the mandatory security awareness course in the Learning Portal within your first 3 days.
- Join the team Slack channels listed in your welcome email.' metadata={'h1': 'Engineering Onboarding', 'h2': 'Day one', 'start_index': 0}

page_content='You will receive these accounts on day one: company email, Slack, GitHub, Jira, Confluence and the internal Zomato Admin dashboard. Production access is **not** granted automatically. Request it through the Access Portal after you finish the security course; your manager must approve it, and access is reviewed every quarter.' metadata={'h1': 'Engineering Onboarding', 'h2': 'Accounts and access', '

In [24]:
from langchain_text_splitters import HTMLHeaderTextSplitter

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=[("h1", "title"), ("h2", "section")])
parts = html_splitter.split_text(open("knowledge/wiki/holiday_calendar.html", encoding="utf-8").read())

for part in parts:
    print(f"{part}\n")

page_content='Company holidays' metadata={'title': 'Company holidays'}

page_content='This calendar applies to all corporate offices. Customer Support and Delivery Operations follow a reduced rota on holidays, and employees who work on a holiday receive a compensatory off.' metadata={'title': 'Company holidays'}

page_content='Fixed-date holidays' metadata={'title': 'Company holidays', 'section': 'Fixed-date holidays'}

page_content='Holiday  
Date  
Republic Day  
26 January  
Independence Day  
15 August  
Gandhi Jayanti  
2 October  
Christmas Day  
25 December' metadata={'title': 'Company holidays', 'section': 'Fixed-date holidays'}

page_content='Festival holidays' metadata={'title': 'Company holidays', 'section': 'Festival holidays'}

page_content='Holi, Eid and Diwali dates follow the lunar calendar. People Operations publishes the exact dates every December. Offices stay closed on these days. Diwali is a two-day holiday.' metadata={'title': 'Company holidays', 'section': 'Festi

## Splitting code and JSON

In [25]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

source = open("knowledge/wiki/retry_helper.py", encoding="utf-8").read()

code_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=600,
    chunk_overlap=60,
)
code_chunks = code_splitter.create_documents([source])
print(len(code_chunks), "chunks, each ending near a function boundary")

6 chunks, each ending near a function boundary


In [26]:
import json
from langchain_text_splitters import RecursiveJsonSplitter

data = json.load(open("knowledge/catalog/plans.json", encoding="utf-8"))
print(data)

json_splitter = RecursiveJsonSplitter(max_chunk_size=400)
json_docs = json_splitter.create_documents(texts=[data])
print(len(json_docs), "JSON chunks")

for chunk in json_docs:
    print(f"{chunk}\n")

{'product': 'Zomato for Enterprise', 'currency': 'INR', 'billing': {'cycle': 'monthly', 'invoice': 'one consolidated GST invoice', 'payment_terms_days': 30, 'annual_commitment_discount_percent': 10}, 'plans': [{'name': 'Starter', 'price_per_employee_per_month': 99, 'min_employees': 10, 'sso': False, 'scim': False, 'support': 'Email support, response within 2 business days', 'features': ['Company meal wallet', 'Daily spending limits', 'Monthly consolidated invoice', 'Email support']}, {'name': 'Business', 'price_per_employee_per_month': 199, 'min_employees': 50, 'sso': True, 'scim': False, 'support': 'Chat support, 9am to 9pm, response within 4 hours', 'features': ['Everything in Starter', 'Single sign-on (SAML 2.0)', 'Department-wise reports', 'Allowed time windows and restaurant rules', 'Scheduled usage reports by email', 'Chat support, 9am to 9pm']}, {'name': 'Enterprise', 'price_per_employee_per_month': 349, 'min_employees': 500, 'sso': True, 'scim': True, 'support': '24x7 priority 

## Semantic chunking

In [27]:
%pip install langchain-experimental

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

semantic = SemanticChunker(
    OpenAIEmbeddings(model="text-embedding-3-small"),
    breakpoint_threshold_type="percentile",     # cut at the biggest drops in similarity
)

chunks = semantic.split_documents(docs[:5])
print(len(chunks), "semantic chunks")

## One chunking module for every document type


In [29]:
"""Turn Documents into chunks, using the splitter that suits each document type."""
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

_by_size = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=120, add_start_index=True)
_by_header = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")])


def _split_markdown(doc: Document) -> list[Document]:
    sections = _by_header.split_text(doc.page_content)
    for section in sections:
        section.metadata = {**doc.metadata, **section.metadata}   # keep file info, add headings
    return _by_size.split_documents(sections)


def chunk_corpus(docs: list[Document]) -> list[Document]:
    chunks: list[Document] = []
    for doc in docs:
        kind = doc.metadata["doc_type"]
        if kind == "md":
            chunks.extend(_split_markdown(doc))
        elif kind == "csv":
            chunks.append(doc)                       # a row is already one answer
        else:
            chunks.extend(_by_size.split_documents([doc]))
    print(f"{len(docs)} documents became {len(chunks)} chunks")
    return chunks

## Metadata and stable chunk IDs


In [30]:
import hashlib
import uuid
from langchain_core.documents import Document

def generate_stable_id(chunk: Document) -> str:
    """
    Generates a deterministic ID based on the file name and chunk content.
    """
    # 1. Get source filename (or fallback to 'unknown')
    filename = chunk.metadata.get("filename", "unknown")
    
    # 2. Hash the chunk's text content
    text_hash = hashlib.sha1(chunk.page_content.encode("utf-8")).hexdigest()
    
    # 3. Combine filename and text hash into a deterministic UUID (v5)
    unique_string = f"{filename}|{text_hash}"
    return str(uuid.uuid5(uuid.NAMESPACE_URL, unique_string))


# --- Example Usage ---

# Create dummy chunks
chunk1 = Document(
    page_content="Full-time employees receive 24 days of paid annual leave.",
    metadata={"filename": "employee_handbook.pdf", "page": 1}
)

chunk2 = Document(
    page_content="Up to 5 unused days may be carried over into the next year.",
    metadata={"filename": "employee_handbook.pdf", "page": 1}
)

# Generate IDs
id1_run1 = generate_stable_id(chunk1)
id2_run1 = generate_stable_id(chunk2)

print("First Run IDs:")
print(f"Chunk 1 ID: {id1_run1}")
print(f"Chunk 2 ID: {id2_run1}")

# Simulate re-running ingestion on Chunk 1
id1_run2 = generate_stable_id(chunk1)

print("\nSecond Run ID (Re-ingesting Chunk 1):")
print(f"Chunk 1 ID: {id1_run2}")

# Verify stability
print(f"\nIs the ID identical across runs? {id1_run1 == id1_run2}")

First Run IDs:
Chunk 1 ID: c67646ad-2e2c-5e60-b74b-56d731672fa6
Chunk 2 ID: 8ed017e3-a930-5936-a3f2-cad3f970e6a2

Second Run ID (Re-ingesting Chunk 1):
Chunk 1 ID: c67646ad-2e2c-5e60-b74b-56d731672fa6

Is the ID identical across runs? True


# Vectors and the vector store

## Embeddings in LangChain

In [4]:
from pydantic import SecretStr
from langchain_openai import OpenAIEmbeddings
import os

embeddings = OpenAIEmbeddings(
    model="nvidia/nemotron-3-embed-1b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=SecretStr(os.environ["OPENROUTER_API_KEY"]),
    check_embedding_ctx_length=False,
)

texts = [
    "Full-time employees receive 24 days of paid annual leave.",
    "Client meals are reimbursed up to the per-person limit with a receipt.",
    "Restart the payments service from the runbook, never from the console.",
]

vectors = embeddings.embed_documents(texts=texts)

print(f"Number of vectors: {len(vectors)}")
print(f"Vector dimension: {len(vectors[0])}")
print(f"First 5 vavlues: {vectors[0][:5]}")

query_vector = embeddings.embed_query("How many vacation days do I get ?")
print("Query vector dimesion:", len(query_vector))

Number of vectors: 3
Vector dimension: 2048
First 5 vavlues: [0.11108508706092834, -0.0035425054375082254, -0.05453040078282356, 0.008626926690340042, -0.004991305060684681]
Query vector dimesion: 2048


In [ ]:
# Option 1: hosted API (1536 dimensions)
# from langchain_openai import OpenAIEmbeddings
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Option 2: Using OpenRouter
# embeddings = OpenAIEmbeddings(
#     model="openai/text-embedding-3-small",
#     base_url="https://openrouter.ai/api/v1",
#     api_key=SecretStr(os.environ["OPENROUTER_API_KEY"]),
# )

# Option 3: a free local model (384 dimensions), no API key
# pip install langchain-huggingface sentence-transformers
# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

## Qdrant through LangChain


### Build the vector store

In [34]:
from langchain_qdrant import QdrantVectorStore

vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=ids,
    location=":memory:",
    collection_name="helpdesk_kb"
)

### Persisting data and running a server

#### Run Qdrant in Docker

docker run -p 6333:6333 -v "$(pwd)/qdrant_storage:/qdrant/storage" qdrant/qdrant

### Full control over the collection

In [ ]:
from qdrant_client import QdrantClient, models
from langchain_qdrant import QdrantVectorStore
import uuid

client = QdrantClient("http://localhost:6333")
dimension = len(vectors[0])

if not client.collection_exists("helpdesk_kb"):
    client.create_collection(
        collection_name="helpdesk_kb",
        vectors_config=models.VectorParams(size=dimension, distance=models.Distance.COSINE)
    )

doc_embeddings = OpenAIEmbeddings(
    model="nvidia/nemotron-3-embed-1b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=SecretStr(os.environ["OPENROUTER_API_KEY"]),
    check_embedding_ctx_length=False,
)

vector_store = QdrantVectorStore(client=client, collection_name="helpdesk_kb", embedding=doc_embeddings)

# Add a small batch of docs to the Qdrant collection
added_ids = vector_store.add_documents(
    documents=docs,
    ids=[str(uuid.uuid4()) for _ in docs],
)

print(f"Added {len(added_ids)} points to Qdrant")

print(f"Added {len(added_ids)} points to Qdrant")
print(vector_store.collection_name)

Added 134 points to Qdrant
helpdesk_kb


### Similarity search

In [35]:
question = "How many vacations do I get ?"

results = vector_store.similarity_search(query=question, k=3)

print(results)
for rank, doc in enumerate(results, start=1):
    print(f"\n--- Result {rank} ---\n")
    print(doc.metadata["filename"], doc.metadata.get("page"))
    print(doc.page_content)

for doc, score in vector_store.similarity_search_with_score(question, k=3):
    print(round(score, 3), doc.metadata["filename"], doc.metadata.get("page"))

[Document(metadata={'producer': 'Skia/PDF m154', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/154.0.0.0 Safari/537.36', 'creationdate': '2026-10-02T18:18:26+00:00', 'source': 'D:\\coding shuttle\\ai engineer\\practice-lab\\ai-eng\\src\\ai_eng\\4_langchain_rag\\knowledge\\policies\\employee_handbook.pdf', 'file_path': 'D:\\coding shuttle\\ai engineer\\practice-lab\\ai-eng\\src\\ai_eng\\4_langchain_rag\\knowledge\\policies\\employee_handbook.pdf', 'total_pages': 8, 'format': 'PDF 1.4', 'title': 'handbook.html', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-10-02T18:18:26+00:00', 'trapped': '', 'modDate': "D:20261002181826+00'00'", 'creationDate': "D:20261002181826+00'00'", 'page': 1, 'filename': 'employee_handbook.pdf', 'area': 'policies', 'doc_type': 'pdf', '_id': '765ae8f0-8a30-4fe2-940f-f9aa78aeeb8c', '_collection_name': 'helpdesk_kb'}, page_content='4. Working hours\nStandard hours are 10:00 to 19:00

### From vector store to retriever

In [3]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}
)

result_docs = retriever.invoke("what is the process for reimbusing a client dinner ?")

for doc in docs:
    print(doc.metadata["filename"], doc.metadata.get("page"))

NameError: name 'vector_store' is not defined

### Metadata filtering

In [40]:
from qdrant_client import models

only_policies = models.Filter(
    must=[
        models.FieldCondition(
            key="metadata.area",                      # LangChain stores metadata under "metadata"
            match=models.MatchValue(value="policies"),
        )
    ]
)

results = vector_store.similarity_search(
    "Can I expense a client dinner?",
    k=4,
    filter=only_policies,
)

for doc in results:
    print(doc.metadata["filename"], doc.metadata.get("page"))

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1791763200000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.'}}}

# Retrieve, prompt and answer (Generation)


## The chat model

In [ ]:
from langchain_openrouter import ChatOpenRouter

llm  = ChatOpenRouter(
    model="poolside/laguna-s-2.1:free",
    temperature=0
)

response = llm.invoke([
    ("system","You are a concise assistant"),
    ("user","explain RAG in one sentence")
])

print(response.content)

In [45]:
%pip install langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain_openai import ChatOpenAI
from pydantic import SecretStr
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(
    base_url="https://api.flex.ai/v1",
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"]),
    model="gpt-oss-120b",
    temperature=0
)

response = llm.invoke([
    ("system","You are a concise assistant"),
    ("user","explain RAG in one sentence")
])

print(response.content)

Retrieval‑Augmented Generation (RAG) combines a language model with a searchable external knowledge base, first retrieving relevant documents and then using them to guide the model’s generated response.


## The RAG prompt


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
    """You are Zomato's internal help-desk assistant.
Answer using only the context below.

Rules:
- If the answer is not in the context, say you don't know.
- Cite the sources you used as [1], [2], matching the numbers in the context.
- Never invent policies, amounts or procedures.

Context:
{context}"""
     ),
    ("user","{question}")
])



## Output parsers

### StrOutputParser: just the text

In [57]:
from langchain_core.output_parsers import StrOutputParser

message = llm.invoke("Name one benefit of RAG in 5 words")
print(type(message).__name__)
print(message.content)

chain = llm | StrOutputParser()
answer = chain.invoke("Name one benefit of RAG in 5 words")
print(type(answer).__name__)
print(message.content)

AIMessage
One benefit of RAG (Rapid Application Development) is that it enables faster time-to-market.
TextAccessor
One benefit of RAG (Rapid Application Development) is that it enables faster time-to-market.


### CommaSeparatedListOutputParser: a real Python list


In [63]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import ChatPromptTemplate

list_parser = CommaSeparatedListOutputParser()

keyword_prompt = ChatPromptTemplate.from_template(
    "Give 3 short search keywords for this question.\n{format_instructions}\nQuestion: {question}"
).partial(format_instructions=list_parser.get_format_instructions())

keyword_chain = keyword_prompt | llm | list_parser

print(keyword_chain.invoke("How many days of paid leave do I get?"))

['paid leave days', 'vacation entitlement', 'annual leave policy']


### JsonOutputParser: a dictionary you can branch on

In [64]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

route_prompt = ChatPromptTemplate.from_template(
    "Decide which area of the Zomato knowledge base best answers the question. "
    'Reply with JSON containing "area" (policies, wiki, faq or catalog) and a short "reason".\n'
    "{format_instructions}\nQuestion: {question}"
).partial(format_instructions=json_parser.get_format_instructions())

route_chain = route_prompt | llm | json_parser

decision = route_chain.invoke("How do I restart the payments service?")
print(decision["area"])         # wiki

wiki


### PydanticOutputParser: a typed, validated object

In [65]:
from typing import Literal

from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


class QueryPlan(BaseModel):
    area: Literal["policies", "wiki", "faq", "catalog"] = Field(description="Where the answer lives")
    topic: str = Field(description="The topic in 2 or 3 words")
    needs_exact_numbers: bool = Field(description="True if amounts or dates are wanted")


plan_parser = PydanticOutputParser(pydantic_object=QueryPlan)

plan_prompt = ChatPromptTemplate.from_template(
    "Analyse the question.\n{format_instructions}\nQuestion: {question}"
).partial(format_instructions=plan_parser.get_format_instructions())

plan_chain = plan_prompt | llm | plan_parser

plan = plan_chain.invoke("What is the limit for client dinners?")

print(type(plan).__name__)          # QueryPlan, a real object
print(plan.area)                    # policies
print(plan.topic)                   # client dinner limit
print(plan.needs_exact_numbers)     # True

QueryPlan
policies
client dinner limit
True


## The explicit RAG pipeline

In [1]:
def cite(doc) -> str:
    m = doc.metadata
    if "page" in m:
        return f"{m['filename']}, page {m['page'] + 1}"           # PDF: page is zero-based
    section = m.get("h3") or m.get("h2") or m.get("h1")
    if section:
        return f"{m['filename']}, section '{section}'"            # Markdown: heading path
    return m["filename"]                                          # CSV and others


def format_docs(docs) -> str:
    """Number each chunk and label it with its origin so the LLM can cite it."""
    return "\n\n---\n\n".join(
        f"[{i}] ({cite(d)})\n{d.page_content}" for i, d in enumerate(docs, start=1)
    )

In [ ]:
question = "How many days of paid leave do full-time employees get?"

docs = retriever.invoke(question)                                   # 1. retrieve
context = format_docs(docs)                                         # 2. format
messages = prompt.invoke({"context": context, "question": question})  # 3. build the prompt
response = llm.invoke(messages)                                     # 4. generate

print(response.content)

## The same pipeline as an LCEL chain

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever | format_docs,   # question -> Documents -> context string
        "question": RunnablePassthrough(),    # question -> question, unchanged
    }
    | prompt
    | llm
    | StrOutputParser()                       # AIMessage -> plain string
)

print(rag_chain.invoke("Which plan includes single sign-on?"))

In [ ]:
# stream tokens as they are generated
for token in rag_chain.stream("How do I set up the company VPN?"):
    print(token, end="", flush=True)

# answer several questions at once
answers = rag_chain.batch([
    "How many days of paid leave do I get?",
    "Can I expense a client dinner?",
])

## Debugging a RAG pipeline

In [ ]:
question = "How do I restart the payments service?"

docs = retriever.invoke(question)
print("RETRIEVED")
for i, d in enumerate(docs, start=1):
    print(f"[{i}] {cite(d)}: {d.page_content[:120]!r}")

context = format_docs(docs)
print("\nCONTEXT LENGTH:", len(context), "characters")

messages = prompt.invoke({"context": context, "question": question})
print("\nPROMPT MESSAGES")
for m in messages.to_messages():
    print(f"- {m.type}: {m.content[:100]!r}")

print("\nANSWER")
print(llm.invoke(messages).content)